In [17]:
# ============================================================
# CELL 1: Check GPU & install YOLOv8
# ============================================================
!nvidia-smi
!pip install ultralytics -q

Sun Mar 29 13:55:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             34W /   70W |    6061MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ============================================================
# VERIFY: GPU + Internet + Dataset paths
# ============================================================
import os

# 1. Check GPU
print("=" * 50)
print("🖥️ GPU STATUS:")
!nvidia-smi | head -15

# 2. Check ultralytics installed
print("\n" + "=" * 50)
print("📦 ULTRALYTICS:")
import ultralytics
print(f"✅ Version: {ultralytics.__version__}")

# 3. Check exact dataset paths
print("\n" + "=" * 50)
print("📁 DATASET STRUCTURE:")
for split in ['train', 'val']:
    for folder in ['images', 'annotations']:
        # Try all possible path variations
        paths_to_try = [
            f'/kaggle/input/mesad-p/mesad-phantom/mesad-phantom/{split}/{folder}',
            f'/kaggle/input/mesad-p/mesad-phantom/{split}/{folder}',
            f'/kaggle/input/mesad-p/{split}/{folder}',
        ]
        for path in paths_to_try:
            if os.path.exists(path):
                count = len(os.listdir(path))
                print(f"✅ {path}")
                print(f"   → {count} files")
                break
        else:
            print(f"❌ {split}/{folder} — path not found")

# 4. Print full input tree
print("\n" + "=" * 50)
print("📂 FULL INPUT TREE:")
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    if level < 5:
        print('  ' * level + os.path.basename(root) + '/')

🖥️ GPU STATUS:
Sat Mar 28 08:00:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------------

In [4]:
# Find exact paths
import os

def find_path(root, target_folder, max_depth=8):
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.replace(root, '').count(os.sep)
        if depth > max_depth:
            continue
        if os.path.basename(dirpath) == target_folder:
            print(f"✅ FOUND: {dirpath} → {len(os.listdir(dirpath))} items")

print("🔍 Searching for 'train' folders...")
find_path('/kaggle/input', 'train')

print("\n🔍 Searching for 'images' folders...")
find_path('/kaggle/input', 'images')

print("\n🔍 Searching for 'annotations' folders...")
find_path('/kaggle/input', 'annotations')

🔍 Searching for 'train' folders...
✅ FOUND: /kaggle/input/datasets/srkr05/mesad-p/mesad-phantom/mesad-phantom/train → 2 items
✅ FOUND: /kaggle/input/datasets/srkr05/mesad-p/mesad-real/mesad-real/train → 2 items

🔍 Searching for 'images' folders...
✅ FOUND: /kaggle/input/datasets/srkr05/mesad-p/mesad-phantom/mesad-phantom/val/images → 2285 items
✅ FOUND: /kaggle/input/datasets/srkr05/mesad-p/mesad-phantom/mesad-phantom/train/images → 22609 items
✅ FOUND: /kaggle/input/datasets/srkr05/mesad-p/mesad-real/mesad-real/val/images → 2024 items
✅ FOUND: /kaggle/input/datasets/srkr05/mesad-p/mesad-real/mesad-real/train/images → 23366 items

🔍 Searching for 'annotations' folders...
✅ FOUND: /kaggle/input/datasets/srkr05/mesad-p/mesad-phantom/mesad-phantom/val/annotations → 4570 items
✅ FOUND: /kaggle/input/datasets/srkr05/mesad-p/mesad-phantom/mesad-phantom/train/annotations → 45218 items
✅ FOUND: /kaggle/input/datasets/srkr05/mesad-p/mesad-real/mesad-real/val/annotations → 4048 items
✅ FOUND: /k

In [6]:
# ============================================================
# CELL 2: Class map — CORRECTED PATH
# ============================================================
import os, pandas as pd, yaml
from PIL import Image
from ultralytics import YOLO

# ✅ CORRECTED — srkr05 instead of mesad2026
dataset_root = '/kaggle/input/datasets/srkr05/mesad-p/mesad-phantom/mesad-phantom'
output_root  = '/kaggle/working/dataset'

def collect_classes(split):
    ann_path = f'{dataset_root}/{split}/annotations'
    classes  = set()
    for f in os.listdir(ann_path):
        if f.endswith('.bboxes.labels.tsv'):
            try:
                df = pd.read_csv(f'{ann_path}/{f}', sep='\t', header=None)
                for val in df[0].values:
                    classes.add(str(val).strip())
            except: pass
    return classes

all_classes = sorted(collect_classes('train') | collect_classes('val'))
class_to_id = {c: i for i, c in enumerate(all_classes)}
print(f"✅ {len(all_classes)} classes: {all_classes}")

✅ 14 classes: ['BladderAnastomosis', 'BladderNeckDissection', 'CuttingSeminalVesicle', 'CuttingTissue', 'CuttingVasDeferens', 'GraspingCatheter', 'MovingDownBladder', 'PassingNeedle', 'PullingBladderNeck', 'PullingProstate', 'PullingSeminalVesicle', 'PullingTissue', 'PullingVasDeferens', 'UrethraDissection']


In [7]:
# ============================================================
# CELL 3: Convert TSV → YOLO
# ============================================================
def convert_split(split):
    ann_path  = f'{dataset_root}/{split}/annotations'
    img_path  = f'{dataset_root}/{split}/images'
    out_label = f'{output_root}/{split}/labels'
    out_image = f'{output_root}/{split}/images'
    os.makedirs(out_label, exist_ok=True)
    os.makedirs(out_image, exist_ok=True)

    bbox_files = sorted([f for f in os.listdir(ann_path)
                         if f.endswith('.bboxes.tsv')
                         and not f.endswith('.labels.tsv')])
    converted, skipped = 0, 0

    for bbox_fname in bbox_files:
        base        = bbox_fname.replace('.bboxes.tsv', '')
        label_fpath = f'{ann_path}/{base}.bboxes.labels.tsv'
        img_fpath   = f'{img_path}/{base}.jpg'

        if not os.path.exists(label_fpath) or not os.path.exists(img_fpath):
            skipped += 1; continue
        try:
            bbox_df  = pd.read_csv(f'{ann_path}/{bbox_fname}', sep='\t', header=None)
            label_df = pd.read_csv(label_fpath, sep='\t', header=None)
            if len(bbox_df) != len(label_df):
                skipped += 1; continue

            with Image.open(img_fpath) as img:
                W, H = img.width, img.height

            lines = []
            for i in range(len(bbox_df)):
                x1,y1,x2,y2 = (bbox_df.iloc[i][0], bbox_df.iloc[i][1],
                                bbox_df.iloc[i][2], bbox_df.iloc[i][3])
                cls_name = str(label_df.iloc[i][0]).strip()
                cls_id   = class_to_id.get(cls_name, -1)
                if cls_id == -1: continue
                cx = max(0, min(1, ((x1+x2)/2)/W))
                cy = max(0, min(1, ((y1+y2)/2)/H))
                w  = max(0, min(1, (x2-x1)/W))
                h  = max(0, min(1, (y2-y1)/H))
                lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

            with open(f'{out_label}/{base}.txt', 'w') as f:
                f.write('\n'.join(lines))

            out_img = f'{out_image}/{base}.jpg'
            if not os.path.exists(out_img):
                os.symlink(img_fpath, out_img)
            converted += 1

        except Exception as e:
            skipped += 1
            if skipped <= 3:  # show first 3 errors only
                print(f"⚠️ Error on {bbox_fname}: {e}")

        if converted % 2000 == 0 and converted > 0:
            print(f"   {split}: {converted} done...")

    print(f"✅ {split}: {converted} converted, {skipped} skipped")

print("🔄 Converting train...")
convert_split('train')
print("🔄 Converting val...")
convert_split('val')

🔄 Converting train...
   train: 2000 done...
   train: 4000 done...
   train: 6000 done...
   train: 8000 done...
   train: 10000 done...
   train: 12000 done...
   train: 14000 done...
   train: 16000 done...
   train: 18000 done...
   train: 20000 done...
   train: 22000 done...
✅ train: 22609 converted, 0 skipped
🔄 Converting val...
   val: 2000 done...
✅ val: 2285 converted, 0 skipped


In [8]:
# ============================================================
# CELL 4: Write data.yaml
# ============================================================
os.makedirs(output_root, exist_ok=True)
with open(f'{output_root}/data.yaml', 'w') as f:
    yaml.dump({
        'path'  : output_root,
        'train' : 'train/images',
        'val'   : 'val/images',
        'nc'    : len(all_classes),
        'names' : all_classes
    }, f, default_flow_style=False)
print("✅ data.yaml written")
print(open(f'{output_root}/data.yaml').read())

✅ data.yaml written
names:
- BladderAnastomosis
- BladderNeckDissection
- CuttingSeminalVesicle
- CuttingTissue
- CuttingVasDeferens
- GraspingCatheter
- MovingDownBladder
- PassingNeedle
- PullingBladderNeck
- PullingProstate
- PullingSeminalVesicle
- PullingTissue
- PullingVasDeferens
- UrethraDissection
nc: 14
path: /kaggle/working/dataset
train: train/images
val: val/images



In [9]:
# ============================================================
# CELL 5: Train YOLOv8s — Step 2
# ============================================================
from ultralytics import YOLO   # ← add this line
import os

model = YOLO('yolov8s.pt')

model.train(
    data        = '/kaggle/working/dataset/data.yaml',
    epochs      = 50,
    imgsz       = 640,
    batch       = 16,
    name        = 'mesad_phantom_s',
    project     = '/kaggle/working/runs',
    device      = '0,1',
    workers     = 4,
    patience    = 20,
    exist_ok    = True,
    cos_lr      = True,
    mixup       = 0.1,
    copy_paste  = 0.1,
    save_period = 5,
)

Ultralytics 8.4.30 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=mesad_phantom_s, nbs=64, nms=F

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:75: UserWarning: Specified kernel cache directory could not be created! This disables kernel caching. Specified directory is /root/.cache/torch/kernels. This warning will appear only once per process. (Triggered internally at /pytorch/aten/src/ATen/native/cuda/jit_utils.cpp:1487.)
  inter = (torch.min(a2, b2) - torch.max(a1, b1)).clamp_(0).prod(2)


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 6.6it/s 10.8s0.1s
                   all       2285       4213      0.458      0.136      0.104     0.0364

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50      2.43G      1.739      1.492      1.555          1        640: 100% ━━━━━━━━━━━━ 1414/1414 4.4it/s 5:24<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 7.2it/s 10.1s.1s
                   all       2285       4213      0.523      0.123     0.0936     0.0315

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      2.43G      1.758      1.468      1.551          6        640: 100% ━━━━━━━━━━━━ 1414/1414 4.7it/s 5:02<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 7.3it/s 9.9s0.1s
                   a

In [10]:
# ============================================================
# CELL 6: Save best_phantom_s.pt NOW before session expires
# ============================================================
import shutil, os
os.makedirs('/kaggle/working/output', exist_ok=True)

shutil.copy(
    '/kaggle/working/runs/mesad_phantom_s/weights/best.pt',
    '/kaggle/working/output/best_phantom_s.pt'
)
print("✅ best_phantom_s.pt saved — download from Output panel NOW!")

✅ best_phantom_s.pt saved — download from Output panel NOW!


In [11]:
# ============================================================
# STEP 3 CELL 1: Build class map for Real dataset
# ============================================================
import os, pandas as pd, yaml
from PIL import Image
from ultralytics import YOLO

real_root   = '/kaggle/input/datasets/srkr05/mesad-p/mesad-real/mesad-real'
real_output = '/kaggle/working/dataset_real'

def collect_classes_real(split):
    ann_path = f'{real_root}/{split}/annotations'
    classes  = set()
    for f in os.listdir(ann_path):
        if f.endswith('.bboxes.labels.tsv'):
            try:
                df = pd.read_csv(f'{ann_path}/{f}', sep='\t', header=None)
                for val in df[0].values:
                    classes.add(str(val).strip())
            except: pass
    return classes

real_classes = sorted(collect_classes_real('train') | collect_classes_real('val'))
real_cls2id  = {c: i for i, c in enumerate(real_classes)}
print(f"✅ Real dataset: {len(real_classes)} classes:")
for i, c in enumerate(real_classes):
    print(f"   {i}: {c}")

✅ Real dataset: 21 classes:
   0: BaggingProstate
   1: BladderAnastomosis
   2: BladderNeckDissection
   3: ClippingBladderNeck
   4: ClippingSeminalVesicle
   5: ClippingTissue
   6: ClippingVasDeferens
   7: CuttingMesocolon
   8: CuttingProstate
   9: CuttingSeminalVesicle
   10: CuttingThread
   11: CuttingTissue
   12: CuttingVasDeferens
   13: PullingBladderNeck
   14: PullingProstate
   15: PullingSeminalVesicle
   16: PullingTissue
   17: PullingVasDeferens
   18: SuckingBlood
   19: SuckingSmoke
   20: UrethraDissection


In [12]:
# ============================================================
# STEP 3 CELL 2: Convert Real TSV → YOLO
# ============================================================
def convert_real_split(split):
    ann_path  = f'{real_root}/{split}/annotations'
    img_path  = f'{real_root}/{split}/images'
    out_label = f'{real_output}/{split}/labels'
    out_image = f'{real_output}/{split}/images'
    os.makedirs(out_label, exist_ok=True)
    os.makedirs(out_image, exist_ok=True)

    bbox_files = sorted([f for f in os.listdir(ann_path)
                         if f.endswith('.bboxes.tsv')
                         and not f.endswith('.labels.tsv')])
    converted, skipped = 0, 0

    for bbox_fname in bbox_files:
        base        = bbox_fname.replace('.bboxes.tsv', '')
        label_fpath = f'{ann_path}/{base}.bboxes.labels.tsv'
        img_fpath   = f'{img_path}/{base}.jpg'

        if not os.path.exists(label_fpath) or not os.path.exists(img_fpath):
            skipped += 1; continue
        try:
            bbox_df  = pd.read_csv(f'{ann_path}/{bbox_fname}', sep='\t', header=None)
            label_df = pd.read_csv(label_fpath, sep='\t', header=None)
            if len(bbox_df) != len(label_df):
                skipped += 1; continue

            with Image.open(img_fpath) as img:
                W, H = img.width, img.height

            lines = []
            for i in range(len(bbox_df)):
                x1,y1,x2,y2 = (bbox_df.iloc[i][0], bbox_df.iloc[i][1],
                                bbox_df.iloc[i][2], bbox_df.iloc[i][3])
                cls_name = str(label_df.iloc[i][0]).strip()
                cls_id   = real_cls2id.get(cls_name, -1)
                if cls_id == -1: continue
                cx = max(0, min(1, ((x1+x2)/2)/W))
                cy = max(0, min(1, ((y1+y2)/2)/H))
                w  = max(0, min(1, (x2-x1)/W))
                h  = max(0, min(1, (y2-y1)/H))
                lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

            with open(f'{out_label}/{base}.txt', 'w') as f:
                f.write('\n'.join(lines))

            out_img = f'{out_image}/{base}.jpg'
            if not os.path.exists(out_img):
                os.symlink(img_fpath, out_img)
            converted += 1

        except Exception as e:
            skipped += 1
            if skipped <= 3:
                print(f"⚠️ {bbox_fname}: {e}")

        if converted % 2000 == 0 and converted > 0:
            print(f"   {split}: {converted} done...")

    print(f"✅ {split}: {converted} converted, {skipped} skipped")

print("🔄 Converting real train...")
convert_real_split('train')
print("🔄 Converting real val...")
convert_real_split('val')

🔄 Converting real train...
   train: 2000 done...
   train: 4000 done...
   train: 6000 done...
   train: 8000 done...
   train: 10000 done...
   train: 12000 done...
   train: 14000 done...
   train: 16000 done...
   train: 18000 done...
   train: 20000 done...
   train: 22000 done...
✅ train: 23366 converted, 0 skipped
🔄 Converting real val...
   val: 2000 done...
✅ val: 2024 converted, 0 skipped


In [13]:
# ============================================================
# STEP 3 CELL 3: Write Real data.yaml + Train
# ============================================================
os.makedirs(real_output, exist_ok=True)
with open(f'{real_output}/data.yaml', 'w') as f:
    yaml.dump({
        'path'  : real_output,
        'train' : 'train/images',
        'val'   : 'val/images',
        'nc'    : len(real_classes),
        'names' : real_classes
    }, f, default_flow_style=False)
print("✅ Real data.yaml:")
print(open(f'{real_output}/data.yaml').read())

✅ Real data.yaml:
names:
- BaggingProstate
- BladderAnastomosis
- BladderNeckDissection
- ClippingBladderNeck
- ClippingSeminalVesicle
- ClippingTissue
- ClippingVasDeferens
- CuttingMesocolon
- CuttingProstate
- CuttingSeminalVesicle
- CuttingThread
- CuttingTissue
- CuttingVasDeferens
- PullingBladderNeck
- PullingProstate
- PullingSeminalVesicle
- PullingTissue
- PullingVasDeferens
- SuckingBlood
- SuckingSmoke
- UrethraDissection
nc: 21
path: /kaggle/working/dataset_real
train: train/images
val: val/images



In [14]:
# ============================================================
# STEP 3 CELL 4: Train YOLOv8s on Real only
# ============================================================
model = YOLO('yolov8s.pt')

model.train(
    data        = f'{real_output}/data.yaml',
    epochs      = 50,
    imgsz       = 640,
    batch       = 16,
    name        = 'mesad_real_s',
    project     = '/kaggle/working/runs',
    device      = '0,1',
    workers     = 4,
    patience    = 20,
    exist_ok    = True,
    cos_lr      = True,
    mixup       = 0.1,
    copy_paste  = 0.1,
    save_period = 5,
)

Ultralytics 8.4.30 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset_real/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=mesad_real_s, nbs=64, nms

In [15]:
# ============================================================
# STEP 3 CELL 5: Save Real weights immediately
# ============================================================
import shutil
shutil.copy(
    '/kaggle/working/runs/mesad_real_s/weights/best.pt',
    '/kaggle/working/output/best_real_s.pt'
)
print("✅ best_real_s.pt saved — download from Output panel NOW!")


✅ best_real_s.pt saved — download from Output panel NOW!


In [3]:
# ============================================================
# STEP 4 CELL 1: Merge Phantom + Real → Combined dataset
# ============================================================
import os, pandas as pd, yaml, shutil
from PIL import Image
from ultralytics import YOLO

# Paths
phantom_root    = '/kaggle/input/datasets/srkr05/mesad-p/mesad-phantom/mesad-phantom'
real_root       = '/kaggle/input/datasets/srkr05/mesad-p/mesad-real/mesad-real'
phantom_output  = '/kaggle/working/dataset'
real_output     = '/kaggle/working/dataset_real'
combined_output = '/kaggle/working/dataset_combined'

# ── Build unified class list from BOTH datasets ──
def collect_classes(root, split):
    ann_path = f'{root}/{split}/annotations'
    classes  = set()
    for f in os.listdir(ann_path):
        if f.endswith('.bboxes.labels.tsv'):
            try:
                df = pd.read_csv(f'{ann_path}/{f}', sep='\t', header=None)
                for val in df[0].values:
                    classes.add(str(val).strip())
            except: pass
    return classes

all_classes = sorted(
    collect_classes(phantom_root, 'train') |
    collect_classes(phantom_root, 'val')   |
    collect_classes(real_root,    'train') |
    collect_classes(real_root,    'val')
)
cls2id = {c: i for i, c in enumerate(all_classes)}

print(f"✅ Combined class list: {len(all_classes)} classes")
for i, c in enumerate(all_classes):
    print(f"   {i}: {c}")

✅ Combined class list: 24 classes
   0: BaggingProstate
   1: BladderAnastomosis
   2: BladderNeckDissection
   3: ClippingBladderNeck
   4: ClippingSeminalVesicle
   5: ClippingTissue
   6: ClippingVasDeferens
   7: CuttingMesocolon
   8: CuttingProstate
   9: CuttingSeminalVesicle
   10: CuttingThread
   11: CuttingTissue
   12: CuttingVasDeferens
   13: GraspingCatheter
   14: MovingDownBladder
   15: PassingNeedle
   16: PullingBladderNeck
   17: PullingProstate
   18: PullingSeminalVesicle
   19: PullingTissue
   20: PullingVasDeferens
   21: SuckingBlood
   22: SuckingSmoke
   23: UrethraDissection


In [4]:
# ============================================================
# STEP 4 CELL 2: Convert BOTH datasets with unified class IDs
# ============================================================
def convert_to_combined(src_root, split, prefix=''):
    ann_path  = f'{src_root}/{split}/annotations'
    img_path  = f'{src_root}/{split}/images'
    out_label = f'{combined_output}/{split}/labels'
    out_image = f'{combined_output}/{split}/images'
    os.makedirs(out_label, exist_ok=True)
    os.makedirs(out_image, exist_ok=True)

    bbox_files = sorted([f for f in os.listdir(ann_path)
                         if f.endswith('.bboxes.tsv')
                         and not f.endswith('.labels.tsv')])
    converted, skipped = 0, 0

    for bbox_fname in bbox_files:
        base        = bbox_fname.replace('.bboxes.tsv', '')
        label_fpath = f'{ann_path}/{base}.bboxes.labels.tsv'
        img_fpath   = f'{img_path}/{base}.jpg'

        if not os.path.exists(label_fpath) or not os.path.exists(img_fpath):
            skipped += 1; continue
        try:
            bbox_df  = pd.read_csv(f'{ann_path}/{bbox_fname}', sep='\t', header=None)
            label_df = pd.read_csv(label_fpath, sep='\t', header=None)
            if len(bbox_df) != len(label_df):
                skipped += 1; continue

            with Image.open(img_fpath) as img:
                W, H = img.width, img.height

            lines = []
            for i in range(len(bbox_df)):
                x1,y1,x2,y2 = (bbox_df.iloc[i][0], bbox_df.iloc[i][1],
                                bbox_df.iloc[i][2], bbox_df.iloc[i][3])
                cls_name = str(label_df.iloc[i][0]).strip()
                cls_id   = cls2id.get(cls_name, -1)
                if cls_id == -1: continue
                cx = max(0, min(1, ((x1+x2)/2)/W))
                cy = max(0, min(1, ((y1+y2)/2)/H))
                w  = max(0, min(1, (x2-x1)/W))
                h  = max(0, min(1, (y2-y1)/H))
                lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

            out_base = f'{prefix}{base}'
            with open(f'{out_label}/{out_base}.txt', 'w') as f:
                f.write('\n'.join(lines))

            out_img = f'{out_image}/{out_base}.jpg'
            if not os.path.exists(out_img):
                os.symlink(img_fpath, out_img)
            converted += 1

        except Exception as e:
            skipped += 1
            if skipped <= 3:
                print(f"⚠️ {bbox_fname}: {e}")

        if converted % 2000 == 0 and converted > 0:
            print(f"   {prefix}{split}: {converted} done...")

    print(f"✅ {prefix}{split}: {converted} converted, {skipped} skipped")

print("🔄 Converting Phantom train...")
convert_to_combined(phantom_root, 'train', prefix='ph_')
print("🔄 Converting Phantom val...")
convert_to_combined(phantom_root, 'val',   prefix='ph_')
print("🔄 Converting Real train...")
convert_to_combined(real_root, 'train',    prefix='re_')
print("🔄 Converting Real val...")
convert_to_combined(real_root, 'val',      prefix='re_')

# Verify counts
for split in ['train', 'val']:
    imgs = len(os.listdir(f'{combined_output}/{split}/images'))
    lbls = len(os.listdir(f'{combined_output}/{split}/labels'))
    print(f"✅ Combined {split}: {imgs} images, {lbls} labels")

🔄 Converting Phantom train...
   ph_train: 2000 done...
   ph_train: 4000 done...
   ph_train: 6000 done...
   ph_train: 8000 done...
   ph_train: 10000 done...
   ph_train: 12000 done...
   ph_train: 14000 done...
   ph_train: 16000 done...
   ph_train: 18000 done...
   ph_train: 20000 done...
   ph_train: 22000 done...
✅ ph_train: 22609 converted, 0 skipped
🔄 Converting Phantom val...
   ph_val: 2000 done...
✅ ph_val: 2285 converted, 0 skipped
🔄 Converting Real train...
   re_train: 2000 done...
   re_train: 4000 done...
   re_train: 6000 done...
   re_train: 8000 done...
   re_train: 10000 done...
   re_train: 12000 done...
   re_train: 14000 done...
   re_train: 16000 done...
   re_train: 18000 done...
   re_train: 20000 done...
   re_train: 22000 done...
✅ re_train: 23366 converted, 0 skipped
🔄 Converting Real val...
   re_val: 2000 done...
✅ re_val: 2024 converted, 0 skipped
✅ Combined train: 45975 images, 45975 labels
✅ Combined val: 4309 images, 4309 labels


In [5]:
# ============================================================
# STEP 4 CELL 3: Write combined data.yaml
# ============================================================
os.makedirs(combined_output, exist_ok=True)
with open(f'{combined_output}/data.yaml', 'w') as f:
    yaml.dump({
        'path'  : combined_output,
        'train' : 'train/images',
        'val'   : 'val/images',
        'nc'    : len(all_classes),
        'names' : all_classes
    }, f, default_flow_style=False)

print("✅ Combined data.yaml:")
print(open(f'{combined_output}/data.yaml').read())

✅ Combined data.yaml:
names:
- BaggingProstate
- BladderAnastomosis
- BladderNeckDissection
- ClippingBladderNeck
- ClippingSeminalVesicle
- ClippingTissue
- ClippingVasDeferens
- CuttingMesocolon
- CuttingProstate
- CuttingSeminalVesicle
- CuttingThread
- CuttingTissue
- CuttingVasDeferens
- GraspingCatheter
- MovingDownBladder
- PassingNeedle
- PullingBladderNeck
- PullingProstate
- PullingSeminalVesicle
- PullingTissue
- PullingVasDeferens
- SuckingBlood
- SuckingSmoke
- UrethraDissection
nc: 24
path: /kaggle/working/dataset_combined
train: train/images
val: val/images



In [ ]:
# ============================================================
# STEP 4 CELL 4: Train YOLOv8s on Combined
# ============================================================
model = YOLO('yolov8s.pt')

model.train(
    data        = f'{combined_output}/data.yaml',
    epochs      = 50,
    imgsz       = 640,
    batch       = 16,
    name        = 'mesad_combined_s',
    project     = '/kaggle/working/runs',
    device      = '0,1',
    workers     = 4,
    patience    = 20,
    exist_ok    = True,
    cos_lr      = True,
    mixup       = 0.1,
    copy_paste  = 0.1,
    save_period = 5,
)

Ultralytics 8.4.30 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset_combined/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=mesad_combined_s, nbs

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:75: UserWarning: Specified kernel cache directory could not be created! This disables kernel caching. Specified directory is /root/.cache/torch/kernels. This warning will appear only once per process. (Triggered internally at /pytorch/aten/src/ATen/native/cuda/jit_utils.cpp:1487.)
  inter = (torch.min(a2, b2) - torch.max(a1, b1)).clamp_(0).prod(2)


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 135/135 5.9it/s 22.9s0.2s
                   all       4309       7978      0.388      0.133     0.0974     0.0331

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50       2.4G      1.786      2.018       1.79         32        640: 64% ━━━━━━━╸──── 1846/2874 4.8it/s 8:37<3:33

In [4]:
# ============================================================
# SAVE WHATEVER IS THERE RIGHT NOW
# ============================================================
import shutil, os

os.makedirs('/kaggle/working/output', exist_ok=True)

# Save best combined weights
best_src = '/kaggle/working/runs/mesad_combined_s/weights/best.pt'
last_src = '/kaggle/working/runs/mesad_combined_s/weights/last.pt'

if os.path.exists(best_src):
    shutil.copy(best_src, '/kaggle/working/output/best_combined_s.pt')
    print("✅ best_combined_s.pt saved!")

if os.path.exists(last_src):
    shutil.copy(last_src, '/kaggle/working/output/last_combined_s.pt')
    print("✅ last_combined_s.pt saved!")

In [2]:
# ============================================================
# FULL RECOVERY — Run this single cell
# Rebuilds everything and starts Step 5 training
# ============================================================
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.transforms.functional import resize, center_crop, normalize
from pathlib import Path
import torch, torch.nn as nn
import os, pandas as pd, cv2, warnings, shutil, yaml
from PIL import Image
from ultralytics import YOLO

warnings.filterwarnings('ignore', category=UserWarning)
torch.use_deterministic_algorithms(False)

# ── Paths ──
phantom_root    = '/kaggle/input/datasets/srkr05/mesad-p/mesad-phantom/mesad-phantom'
real_root       = '/kaggle/input/datasets/srkr05/mesad-p/mesad-real/mesad-real'
combined_output = '/kaggle/working/dataset_combined'
save_dir        = '/kaggle/working/runs/multistream'
out_dir         = '/kaggle/working/output'
os.makedirs(save_dir, exist_ok=True)
os.makedirs(out_dir,  exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

# ── Step 1: Build class list ──
def collect_classes(root, split):
    ann_path = f'{root}/{split}/annotations'
    classes  = set()
    for f in os.listdir(ann_path):
        if f.endswith('.bboxes.labels.tsv'):
            try:
                df = pd.read_csv(f'{ann_path}/{f}', sep='\t', header=None)
                for val in df[0].values: classes.add(str(val).strip())
            except: pass
    return classes

ALL_CLASSES = sorted(
    collect_classes(phantom_root, 'train') |
    collect_classes(phantom_root, 'val')   |
    collect_classes(real_root,    'train') |
    collect_classes(real_root,    'val')
)
NUM_CLASSES = len(ALL_CLASSES)
cls2id = {c: i for i, c in enumerate(ALL_CLASSES)}
print(f"✅ {NUM_CLASSES} classes ready")

# ── Step 2: Convert both datasets ──
def convert_to_combined(src_root, split, prefix=''):
    ann_path  = f'{src_root}/{split}/annotations'
    img_path  = f'{src_root}/{split}/images'
    out_label = f'{combined_output}/{split}/labels'
    out_image = f'{combined_output}/{split}/images'
    os.makedirs(out_label, exist_ok=True)
    os.makedirs(out_image, exist_ok=True)

    bbox_files = sorted([f for f in os.listdir(ann_path)
                         if f.endswith('.bboxes.tsv')
                         and not f.endswith('.labels.tsv')])
    converted = 0
    for bbox_fname in bbox_files:
        base        = bbox_fname.replace('.bboxes.tsv', '')
        label_fpath = f'{ann_path}/{base}.bboxes.labels.tsv'
        img_fpath   = f'{img_path}/{base}.jpg'
        if not os.path.exists(label_fpath) or not os.path.exists(img_fpath):
            continue
        try:
            bbox_df  = pd.read_csv(f'{ann_path}/{bbox_fname}', sep='\t', header=None)
            label_df = pd.read_csv(label_fpath, sep='\t', header=None)
            if len(bbox_df) != len(label_df): continue
            with Image.open(img_fpath) as img:
                W, H = img.width, img.height
            lines = []
            for i in range(len(bbox_df)):
                x1,y1,x2,y2 = (bbox_df.iloc[i][0], bbox_df.iloc[i][1],
                                bbox_df.iloc[i][2], bbox_df.iloc[i][3])
                cls_name = str(label_df.iloc[i][0]).strip()
                cls_id   = cls2id.get(cls_name, -1)
                if cls_id == -1: continue
                cx = max(0, min(1, ((x1+x2)/2)/W))
                cy = max(0, min(1, ((y1+y2)/2)/H))
                w  = max(0, min(1, (x2-x1)/W))
                h  = max(0, min(1, (y2-y1)/H))
                lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            out_base = f'{prefix}{base}'
            with open(f'{out_label}/{out_base}.txt', 'w') as f:
                f.write('\n'.join(lines))
            out_img = f'{combined_output}/{split}/images/{out_base}.jpg'
            if not os.path.exists(out_img):
                os.symlink(img_fpath, out_img)
            converted += 1
        except: pass
        if converted % 3000 == 0 and converted > 0:
            print(f"   {prefix}{split}: {converted} done...")
    print(f"✅ {prefix}{split}: {converted} converted")

print("🔄 Converting datasets...")
convert_to_combined(phantom_root, 'train', 'ph_')
convert_to_combined(phantom_root, 'val',   'ph_')
convert_to_combined(real_root,    'train', 're_')
convert_to_combined(real_root,    'val',   're_')

# Write data.yaml
os.makedirs(combined_output, exist_ok=True)
with open(f'{combined_output}/data.yaml', 'w') as f:
    yaml.dump({'path': combined_output, 'train': 'train/images',
               'val': 'val/images', 'nc': NUM_CLASSES,
               'names': ALL_CLASSES}, f, default_flow_style=False)
print("✅ data.yaml written")

# ── Step 3: Train YOLOv8s combined (Step 4) ──
print("\n🚀 Starting Step 4 — YOLOv8s Combined training...")
yolo_model = YOLO('yolov8s.pt')
yolo_model.train(
    data        = f'{combined_output}/data.yaml',
    epochs      = 30,          # reduced for time
    imgsz       = 640,
    batch       = 16,
    name        = 'mesad_combined_s',
    project     = '/kaggle/working/runs',
    device      = '0,1',
    workers     = 4,
    patience    = 15,
    exist_ok    = True,
    cos_lr      = True,
    mixup       = 0.1,
    copy_paste  = 0.1,
    save_period = 5,
)

# Save YOLOv8s weights immediately
shutil.copy(
    '/kaggle/working/runs/mesad_combined_s/weights/best.pt',
    f'{out_dir}/best_combined_s.pt'
)
print("✅ best_combined_s.pt saved to output — DOWNLOAD NOW!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Device: cuda
✅ 24 classes ready
🔄 Converting datasets...
   ph_train: 3000 done...
   ph_train: 6000 done...
   ph_train: 9000 done...
   ph_train: 12000 done...
   ph_train: 15000 done...
   ph_train: 18000 done...
   ph_train: 21000 done...
✅ ph_train: 22609 converted
✅ ph_val: 2285 converted
   re_train: 3000 done...
   re_train: 6000 done...
   re_train: 9000 done...
   re_train: 12000 done...
   re_train: 15000 done...
   re_train: 18000 done...
   re_train: 21000 done...
✅ re_train: 23366 converted
✅ re_val: 2024 converted
✅ data.yaml written

🚀 Starting Step 4 — YOLOv8s Combined training...
Ultralytics 8.4.31 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

W0329 09:17:04.940000 127 torch/distributed/elastic/agent/server/api.py:739] Received 2 death signal, shutting down workers
W0329 09:17:04.942000 127 torch/distributed/elastic/multiprocessing/api.py:1010] Sending process 130 closing signal SIGINT
W0329 09:17:04.944000 127 torch/distributed/elastic/multiprocessing/api.py:1010] Sending process 131 closing signal SIGINT
[rank0]: Traceback (most recent call last):
[rank0]:   File "/root/.config/Ultralytics/DDP/_temp_pcj5hwlw136247988521968.py", line 19, in <module>
[rank0]:     results = trainer.train()
[rank0]:               ^^^^^^^^^^^^^^^
[rank0]:   File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py", line 244, in train
[rank0]:     self._do_train()
[rank0]:   File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py", line 436, in _do_train
[rank0]:     loss, self.loss_items = self.model(batch)
[rank0]:                             ^^^^^^^^^^^^^^^^^
[rank0]:   File "/usr/local/lib/python3.12/di

KeyboardInterrupt: 

In [4]:
import os

# Find your uploaded weights
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.pt'):
            full_path = os.path.join(root, f)
            print(f"✅ Found: {full_path}")

✅ Found: /kaggle/input/datasets/srkr05/mesad-combined-weights/best_combined_s.pt


KeyboardInterrupt: 

In [5]:
!pip install ultralytics pytorchvideo timm einops -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 3.4 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 52.8 MB/s eta 0:00:00:00:0100:01


In [7]:
import os

print("🔍 Searching for .pt files...")
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.pt') or f.endswith('.pth'):
            print(f"✅ {os.path.join(root, f)}")

🔍 Searching for .pt files...
✅ /kaggle/input/datasets/srkr05/mesad-combined-weights/best_combined_s.pt


In [8]:
# ✅ CORRECTED PATH
YOLO_WEIGHTS = '/kaggle/input/datasets/srkr05/mesad-combined-weights/best_combined_s.pt'


import os
print(f"✅ Weights found: {os.path.exists(YOLO_WEIGHTS)}")


✅ Weights found: True


In [ ]:
# ============================================================
# IMPROVED MULTI-STREAM YOLO + SLOWFAST TRAINING
# ============================================================

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.transforms.functional import resize, center_crop, normalize
from pathlib import Path
import torch, torch.nn as nn
import os, pandas as pd, cv2, warnings, shutil, time
from ultralytics import YOLO
from pytorchvideo.models.hub import slowfast_r50

warnings.filterwarnings('ignore')

# ============================================================
# CONFIG (UPDATED)
# ============================================================

SLOW_FRAMES  = 8
FAST_FRAMES  = 32
CROP_SIZE    = 224
RESIZE_TO    = 256

EPOCHS       = 20        # 🔥 increased
BATCH        = 4         # 🔥 increase if GPU supports (else 2)
LR           = 1e-4      # 🔥 reduced
CLIP_STRIDE  = 16        # 🔥 more data

MEAN         = [0.45, 0.45, 0.45]
STD          = [0.225, 0.225, 0.225]

phantom_root = '/kaggle/input/datasets/srkr05/mesad-p/mesad-phantom/mesad-phantom'
real_root    = '/kaggle/input/datasets/srkr05/mesad-p/mesad-real/mesad-real'
YOLO_WEIGHTS = '/kaggle/input/datasets/srkr05/mesad-combined-weights/best_combined_s.pt'

save_dir     = '/kaggle/working/runs/multistream'
out_dir      = '/kaggle/working/output'
os.makedirs(save_dir, exist_ok=True)
os.makedirs(out_dir,  exist_ok=True)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

# ============================================================
# CLASS COLLECTION
# ============================================================

def collect_classes(root, split):
    ann_path = f'{root}/{split}/annotations'
    classes  = set()
    for f in os.listdir(ann_path):
        if f.endswith('.bboxes.labels.tsv'):
            try:
                df = pd.read_csv(f'{ann_path}/{f}', sep='\t', header=None)
                classes.update(df[0].unique())
            except: pass
    return classes

ALL_CLASSES = sorted(
    collect_classes(phantom_root, 'train') |
    collect_classes(real_root,    'train')
)

NUM_CLASSES = len(ALL_CLASSES)
cls2id      = {c: i for i, c in enumerate(ALL_CLASSES)}

print(f"✅ {NUM_CLASSES} classes found")

# ============================================================
# DATASET
# ============================================================

class MESADClipDataset(Dataset):
    def __init__(self, split='train'):
        self.clips = []
        for root in [phantom_root, real_root]:
            img_dir = Path(f'{root}/{split}/images')
            ann_dir = Path(f'{root}/{split}/annotations')

            seqs = {}
            for f in sorted(img_dir.glob('*.jpg')):
                parts = f.stem.split('_frame_')
                if len(parts) != 2: continue
                seq, num = parts[0], int(parts[1])
                seqs.setdefault(seq, []).append((num, f, ann_dir))

            for seq, frames in seqs.items():
                frames.sort(key=lambda x: x[0])
                for i in range(0, len(frames) - FAST_FRAMES, CLIP_STRIDE):
                    self.clips.append(frames[i:i+FAST_FRAMES])

        print(f"✅ {split}: {len(self.clips)} clips")

    def _load_frame(self, path):
        img = cv2.imread(str(path))
        if img is None:
            return torch.zeros(3, CROP_SIZE, CROP_SIZE)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # 🔥 AUGMENTATION
        if torch.rand(1) > 0.5:
            img = cv2.flip(img, 1)

        t = torch.from_numpy(img).permute(2,0,1).float() / 255.0
        t = resize(t, [RESIZE_TO])
        t = center_crop(t, [CROP_SIZE, CROP_SIZE])
        t = normalize(t, MEAN, STD)
        return t

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        clip = self.clips[idx]
        frames = torch.stack([self._load_frame(f) for _, f, _ in clip])

        slow_idx  = torch.linspace(0, FAST_FRAMES-1, SLOW_FRAMES).long()
        slow_clip = frames[slow_idx].permute(1,0,2,3)
        fast_clip = frames.permute(1,0,2,3)
        mid_frame = frames[FAST_FRAMES//2]

        label = torch.randint(0, NUM_CLASSES, (1,)).item()  # placeholder

        return slow_clip, fast_clip, mid_frame, label

# ============================================================
# MODEL
# ============================================================

class MultiStreamMESAD(nn.Module):
    def __init__(self):
        super().__init__()

        yolo = YOLO(YOLO_WEIGHTS)
        self.yolo_backbone = yolo.model.model[:10]
        self.pool = nn.AdaptiveAvgPool2d(1)

        sf = slowfast_r50(pretrained=True)
        sf.blocks[-1].proj = nn.Identity()
        self.slowfast = sf

        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(512, 8, 512, batch_first=True),
            num_layers=1
        )

        self.fc = nn.Sequential(
            nn.Linear(512+2304, 256),
            nn.GELU(),
            nn.Dropout(0.5),   # 🔥 improved
            nn.Linear(256, NUM_CLASSES)
        )

        # freeze initially
        for p in self.yolo_backbone.parameters():
            p.requires_grad = False

    def forward(self, slow, fast, frame):

        x = frame
        for layer in self.yolo_backbone:
            x = layer(x)
        yolo_feat = self.pool(x).flatten(1)

        sf_feat = self.slowfast([slow, fast]).flatten(1)

        attn = self.transformer(yolo_feat.unsqueeze(1)).squeeze(1)

        fused = torch.cat([attn, sf_feat], dim=1)
        return self.fc(fused)

# ============================================================
# TRAINING
# ============================================================

train_ds = MESADClipDataset('train')
val_ds   = MESADClipDataset('val')

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH)

model = MultiStreamMESAD().to(device)

criterion = nn.CrossEntropyLoss()   # 🔥 fixed
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=5e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_acc = 0

print("\n🚀 Training Started\n")

for epoch in range(EPOCHS):

    # 🔥 UNFREEZE EARLY
    if epoch == 2:
        for p in model.yolo_backbone.parameters():
            p.requires_grad = True
        print("🔓 YOLO unfrozen")

    # TRAIN
    model.train()
    correct, total = 0, 0

    for slow, fast, frame, labels in train_loader:
        slow, fast, frame, labels = slow.to(device), fast.to(device), frame.to(device), labels.to(device)

        optimizer.zero_grad()
        out = model(slow, fast, frame)
        loss = criterion(out, labels)

        loss.backward()
        optimizer.step()

        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total * 100

    # VALIDATION
    model.eval()
    v_correct, v_total = 0, 0

    with torch.no_grad():
        for slow, fast, frame, labels in val_loader:
            slow, fast, frame, labels = slow.to(device), fast.to(device), frame.to(device), labels.to(device)
            out = model(slow, fast, frame)
            v_correct += (out.argmax(1) == labels).sum().item()
            v_total += labels.size(0)

    val_acc = v_correct / v_total * 100
    scheduler.step()

    print(f"Epoch {epoch+1}/{EPOCHS} | Train {train_acc:.1f}% | Val {val_acc:.1f}%")

    # SAVE
    torch.save(model.state_dict(), f"{save_dir}/last.pth")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), f"{save_dir}/best.pth")
        print("⭐ Best model saved!")

print("\n🎉 Training Complete")
print("Best Accuracy:", best_acc)

✅ Device: cuda:0
✅ 24 classes found
✅ train: 2861 clips
✅ val: 266 clips

🚀 Training Started

Epoch 1/20 | Train 3.7% | Val 4.5%
⭐ Best model saved!


[rank1]:[E329 12:17:04.359196624 ProcessGroupNCCL.cpp:688] [Rank 1] Watchdog caught collective operation timeout: WorkNCCL(SeqNum=37514, OpType=BROADCAST, NumelIn=20032, NumelOut=20032, Timeout(ms)=10800000) ran for 10800000 milliseconds before timing out.
[rank1]:[E329 12:17:04.361605297 ProcessGroupNCCL.cpp:820] [Rank 1] Work WorkNCCL(SeqNum=37514, OpType=BROADCAST, NumelIn=20032, NumelOut=20032, Timeout(ms)=10800000) timed out in blocking wait.
[rank1]:[E329 12:17:05.887907274 ProcessGroupNCCL.cpp:749] [Rank 1] Some NCCL operations have failed or timed out. Due to the asynchronous nature of CUDA kernels, subsequent GPU operations might run on corrupted/incomplete data.
[rank1]:[E329 12:17:05.887940335 ProcessGroupNCCL.cpp:763] [Rank 1] To avoid data inconsistency, we are taking the entire process down.
[rank1]: Traceback (most recent call last):
[rank1]:   File "/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/distributed.py", line 2195, in _sync_buffers
[rank1]:     self._

Epoch 2/20 | Train 4.0% | Val 3.4%
🔓 YOLO unfrozen
Epoch 3/20 | Train 4.3% | Val 3.4%


In [13]:
import shutil, os, torch

os.makedirs('/kaggle/working/output', exist_ok=True)

for fname in ['best_multistream.pth', 'last_multistream.pth']:
    src = f'/kaggle/working/runs/multistream/{fname}'
    dst = f'/kaggle/working/output/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        c = torch.load(dst, map_location='cpu')
        print(f"✅ {fname} — epoch {c['epoch']+1} — val: {c['val_acc']:.1f}%")

✅ best_multistream.pth — epoch 1 — val: 39.7%
✅ last_multistream.pth — epoch 8 — val: 38.2%


In [18]:
# ============================================================
# COMPLETE FINE-TUNING CELL
# Upload best_multistream.pth as Kaggle dataset first
# ============================================================
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.transforms.functional import resize, center_crop, normalize
from pathlib import Path
import torch, torch.nn as nn
import os, pandas as pd, cv2, warnings, shutil, time, numpy as np
from ultralytics import YOLO
from pytorchvideo.models.hub import slowfast_r50

warnings.filterwarnings('ignore')
torch.use_deterministic_algorithms(False)

SLOW_FRAMES = 8
FAST_FRAMES = 32
CROP_SIZE   = 224
EPOCHS      = 10       # 10 more epochs
BATCH       = 2
save_dir    = '/kaggle/working/runs/multistream_v2'
out_dir     = '/kaggle/working/output'
os.makedirs(save_dir, exist_ok=True)
os.makedirs(out_dir,  exist_ok=True)

phantom_root = '/kaggle/input/datasets/srkr05/mesad-p/mesad-phantom/mesad-phantom'
real_root    = '/kaggle/input/datasets/srkr05/mesad-p/mesad-real/mesad-real'
YOLO_WEIGHTS = '/kaggle/input/datasets/srkr05/mesad-combined-weights/best_combined_s.pt'
CKPT_PATH    = '/kaggle/input/mesad-multistream/best_multistream.pth'
device       = torch.device('cuda:0')

def collect_classes(root, split):
    ann_path = f'{root}/{split}/annotations'
    classes  = set()
    for f in os.listdir(ann_path):
        if f.endswith('.bboxes.labels.tsv'):
            try:
                df = pd.read_csv(f'{ann_path}/{f}', sep='\t', header=None)
                for val in df[0].values: classes.add(str(val).strip())
            except: pass
    return classes

ALL_CLASSES = sorted(
    collect_classes(phantom_root, 'train') |
    collect_classes(phantom_root, 'val')   |
    collect_classes(real_root,    'train') |
    collect_classes(real_root,    'val')
)
NUM_CLASSES = len(ALL_CLASSES)
cls2id = {c: i for i, c in enumerate(ALL_CLASSES)}
print(f"✅ {NUM_CLASSES} classes")

class MESADClipDataset(Dataset):
    def __init__(self, split='train', clip_stride=32):
        self.clips = []
        for src_root in [phantom_root, real_root]:
            img_dir = Path(f'{src_root}/{split}/images')
            ann_dir = Path(f'{src_root}/{split}/annotations')
            if not img_dir.exists(): continue
            seqs = {}
            for f in sorted(img_dir.glob('*.jpg')):
                parts = f.stem.split('_frame_')
                if len(parts) != 2: continue
                seq, fnum = parts[0], int(parts[1])
                seqs.setdefault(seq, []).append((fnum, f, ann_dir))
            for seq, frames in seqs.items():
                frames.sort(key=lambda x: x[0])
                for i in range(0, len(frames) - FAST_FRAMES, clip_stride):
                    self.clips.append(frames[i:i+FAST_FRAMES])
        print(f"✅ {split}: {len(self.clips)} clips")

    def _load_frame(self, path):
        img = cv2.imread(str(path))
        if img is None:
            return torch.zeros(3, CROP_SIZE, CROP_SIZE)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        t   = torch.from_numpy(img).permute(2,0,1).float() / 255.0
        t   = resize(t, [256])
        t   = center_crop(t, [CROP_SIZE, CROP_SIZE])
        t   = normalize(t, [0.45,0.45,0.45], [0.225,0.225,0.225])
        return t

    def _get_label(self, clip_frames):
        from collections import Counter
        labels = []
        for _, fpath, ann_dir in clip_frames:
            lpath = ann_dir / (fpath.stem + '.bboxes.labels.tsv')
            if lpath.exists():
                try:
                    df = pd.read_csv(lpath, sep='\t', header=None)
                    labels.extend(df[0].tolist())
                except: pass
        if not labels: return 0
        return cls2id.get(Counter(labels).most_common(1)[0][0], 0)

    def __len__(self): return len(self.clips)

    def __getitem__(self, idx):
        clip = self.clips[idx]
        try:
            frames = torch.stack([self._load_frame(f) for _, f, _ in clip])
        except:
            frames = torch.zeros(FAST_FRAMES, 3, CROP_SIZE, CROP_SIZE)
        slow_idx  = torch.linspace(0, FAST_FRAMES-1, SLOW_FRAMES).long()
        return (frames[slow_idx].permute(1,0,2,3),
                frames.permute(1,0,2,3),
                frames[FAST_FRAMES//2],
                self._get_label(clip))

# ✅ Larger val set — stride 8 instead of 64
train_ds = MESADClipDataset('train', clip_stride=32)
val_ds   = MESADClipDataset('val',   clip_stride=8)  # more val clips

class MultiStreamMESAD(nn.Module):
    def __init__(self, num_classes, yolo_path):
        super().__init__()
        yolo = YOLO(yolo_path)
        self.yolo_backbone = yolo.model.model[:10]
        self.yolo_pool     = nn.AdaptiveAvgPool2d(1)
        sf = slowfast_r50(pretrained=False)  # pretrained=False since we load ckpt
        sf.blocks[-1].proj = nn.Identity()
        self.slowfast = sf
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=512, nhead=8, dim_feedforward=512,
                dropout=0.1, batch_first=True), num_layers=1)
        self.fusion = nn.Sequential(
            nn.LayerNorm(512+2304),
            nn.Linear(512+2304, 256),
            nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes))

    def _yolo_fwd(self, x):
        for layer in self.yolo_backbone: x = layer(x)
        return x

    def forward(self, slow, fast, frame):
        yolo_feat = self.yolo_pool(self._yolo_fwd(frame)).flatten(1)
        sf_out    = self.slowfast([slow, fast])
        if isinstance(sf_out, list): sf_out = sf_out[0]
        attn      = self.transformer(yolo_feat.unsqueeze(1)).squeeze(1)
        return self.fusion(torch.cat([attn, sf_out.flatten(1)], 1))

# ── Load model + checkpoint ──
model = MultiStreamMESAD(NUM_CLASSES, YOLO_WEIGHTS).to(device)
ckpt  = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state'])
print(f"✅ Loaded checkpoint — val acc: {ckpt['val_acc']:.1f}%")

# ── Unfreeze ALL parameters ──
for p in model.parameters():
    p.requires_grad = True
print("✅ All parameters unfrozen for full fine-tuning")

# ── Differential learning rates ──
optimizer = AdamW([
    {'params': model.yolo_backbone.parameters(), 'lr': 1e-5},
    {'params': model.slowfast.parameters(),       'lr': 5e-5},
    {'params': model.transformer.parameters(),    'lr': 1e-4},
    {'params': model.fusion.parameters(),         'lr': 1e-4},
], weight_decay=1e-4)

scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=2, pin_memory=True)

best_acc = ckpt['val_acc']
print(f"\n🚀 Fine-tuning for {EPOCHS} epochs | Starting best: {best_acc:.1f}%\n")

for epoch in range(EPOCHS):
    epoch_start = time.time()
    model.train()
    t_loss, correct, total = 0, 0, 0

    for batch_idx, (slow, fast, frame, labels) in enumerate(train_loader):
        slow, fast    = slow.to(device), fast.to(device)
        frame, labels = frame.to(device), labels.to(device)
        optimizer.zero_grad()
        try:
            logits = model(slow, fast, frame)
            loss   = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss  += loss.item()
            correct += (logits.argmax(1) == labels).sum().item()
            total   += labels.size(0)
        except RuntimeError as e:
            print(f"   ⚠️ Skipped: {e}")
            continue

        if (batch_idx+1) % 100 == 0:
            print(f"   [{epoch+1}/{EPOCHS}] Batch {batch_idx+1}/"
                  f"{len(train_loader)} | "
                  f"Acc: {correct/max(total,1)*100:.1f}% | "
                  f"Loss: {t_loss/(batch_idx+1):.4f}")

    model.eval()
    v_correct, v_total = 0, 0
    with torch.no_grad():
        for slow, fast, frame, labels in val_loader:
            slow, fast    = slow.to(device), fast.to(device)
            frame, labels = frame.to(device), labels.to(device)
            try:
                logits     = model(slow, fast, frame)
                v_correct += (logits.argmax(1) == labels).sum().item()
                v_total   += labels.size(0)
            except: continue

    val_acc   = v_correct / max(v_total, 1) * 100
    train_acc = correct   / max(total, 1)   * 100
    elapsed   = (time.time() - epoch_start) / 60
    scheduler.step()

    print(f"\nEpoch {epoch+1:>2}/{EPOCHS} | "
          f"Train {train_acc:.1f}% | Val {val_acc:.1f}% | "
          f"Loss {t_loss/max(len(train_loader),1):.4f} | "
          f"Time {elapsed:.1f}min")

    # Save every epoch
    m_state = model.state_dict()
    ckpt_new = {'epoch': epoch, 'model_state': m_state,
                'val_acc': val_acc, 'class_names': ALL_CLASSES}
    torch.save(ckpt_new, f'{save_dir}/last_v2.pth')
    shutil.copy(f'{save_dir}/last_v2.pth', f'{out_dir}/last_multistream_v2.pth')

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(ckpt_new, f'{save_dir}/best_v2.pth')
        shutil.copy(f'{save_dir}/best_v2.pth', f'{out_dir}/best_multistream_v2.pth')
        print(f"   ⭐ NEW BEST: {val_acc:.1f}% — saved!")
    else:
        print(f"   💾 Epoch {epoch+1} saved (best still {best_acc:.1f}%)")
    print()

print(f"🎉 Fine-tuning complete! Best val accuracy: {best_acc:.1f}%")

✅ 24 classes
✅ train: 1432 clips
✅ val: 531 clips


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/srkr05/mesad-combined-weights/best_combined_s.pt'